## 1. Imports and Workspace Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)

# Set style for visualizations
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

## 2. Load the Dataset

In [ ]:
train_path = "../dataset/fraudTrain.csv"
test_path = "../dataset/fraudTest.csv"

print("Loading datasets...")
df_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)

print(f"Training set shape: {df_train.shape}")
print(f"Testing set shape: {df_test.shape}")

## 3. Data Cleaning and Imbalance Check

In [ ]:
# Checking for missing values
print("Missing values in Training set:\n", df_train.isnull().sum())

# Checking for duplicates
print(f"Duplicate rows in training set: {df_train.duplicated().sum()}")

# Class distribution
class_counts = df_train['is_fraud'].value_counts()
print("\nClass Distribution:")
print(class_counts)
print(f"Fraud rate: {class_counts[1] / len(df_train) * 100:.2f}%")

## 4. Feature Engineering

We will extract temporal features (`hour`, `day_of_week`, `month`), calculate customer `age` from date of birth (`dob`), and compute geographical transaction `distance_km` between the customer (`lat`, `long`) and the merchant (`merch_lat`, `merch_long`) using the Haversine formula.

In [ ]:
def haversine_np(lon1, lat1, lon2, lat2):
    lon1_rad, lat1_rad, lon2_rad, lat2_rad = map(np.radians, [lon1, lat1, lon2, lat2])
    dlon = lon2_rad - lon1_rad
    dlat = lat2_rad - lat1_rad
    a = np.sin(dlat/2.0)**2 + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(dlon/2.0)**2
    c = 2.0 * np.arcsin(np.sqrt(a))
    km = 6367 * c
    return km

def engineer_features(df):
    df = df.copy()
    df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'])
    df['dob'] = pd.to_datetime(df['dob'])
    
    df['hour'] = df['trans_date_trans_time'].dt.hour
    df['day_of_week'] = df['trans_date_trans_time'].dt.dayofweek
    df['month'] = df['trans_date_trans_time'].dt.month
    df['age'] = df['trans_date_trans_time'].dt.year - df['dob'].dt.year
    df['distance_km'] = haversine_np(df['long'], df['lat'], df['merch_long'], df['merch_lat'])
    return df

print("Applying feature engineering...")
df_train_proc = engineer_features(df_train)
df_test_proc = engineer_features(df_test)
print("Done!")

## 5. Feature Selection and Preprocessing

We select informative columns and remove identifiers such as transaction ID (`trans_num`), name details, street addresses, and zip codes. Then we perform Label Encoding for categorical variables and Standard Scaling for numeric features.

In [ ]:
features = [
    "amt", "category", "gender", "state", "city_pop", 
    "age", "hour", "day_of_week", "month", "distance_km", 
    "unix_time", "merchant"
]
target = "is_fraud"

X_train = df_train_proc[features].copy()
y_train = df_train_proc[target]

X_test = df_test_proc[features].copy()
y_test = df_test_proc[target]

# Categorical encoding
categorical_cols = ["category", "gender", "state", "merchant"]
for col in categorical_cols:
    le = LabelEncoder()
    # Fit on combined to avoid missing classes in test set
    combined = pd.concat([X_train[col], X_test[col]], axis=0).astype(str)
    le.fit(combined)
    X_train[col] = le.transform(X_train[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))

# Numerical scaling
numerical_cols = ["amt", "city_pop", "age", "hour", "day_of_week", "month", "distance_km", "unix_time"]
scaler = StandardScaler()
X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])

print("Features scaled and encoded successfully.")

## 6. Model Training & Comparison

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    "Decision Tree": DecisionTreeClassifier(random_state=42, class_weight='balanced', max_depth=10),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced', n_jobs=-1)
}

results = {}

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    
    # Predict
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    # Record metrics
    results[name] = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1 Score": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_prob),
        "y_prob": y_prob,
        "y_pred": y_pred
    }
    
print("All models trained successfully.")

## 7. Performance Visualization

In [ ]:
# Create comparison table
comparison_data = []
for name in results:
    comparison_data.append({
        "Model": name,
        "Accuracy": results[name]["Accuracy"],
        "Precision": results[name]["Precision"],
        "Recall": results[name]["Recall"],
        "F1 Score": results[name]["F1 Score"],
        "ROC-AUC": results[name]["ROC-AUC"]
    })

df_comparison = pd.DataFrame(comparison_data)
print(df_comparison.to_string(index=False))

In [ ]:
# Plotting ROC Curves
plt.figure(figsize=(10, 8))
for name in results:
    fpr, tpr, _ = roc_curve(y_test, results[name]["y_prob"])
    plt.plot(fpr, tpr, label=f"{name} (AUC = {results[name]['ROC-AUC']:.4f})")

plt.plot([0, 1], [0, 1], 'k--', label="Random Guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend()
plt.show()

In [ ]:
# Plotting Confusion Matrices
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for idx, name in enumerate(results):
    cm = confusion_matrix(y_test, results[name]["y_pred"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[idx])
    axes[idx].set_title(f"{name} Confusion Matrix")
    axes[idx].set_xlabel("Predicted")
    axes[idx].set_ylabel("Actual")
plt.tight_layout()
plt.show()